In [13]:
import requests
import time  # optional, to avoid rate limits

# OpenAlex ID of the main paper
main_work_id = "W4406696697"
main_url = f"https://api.openalex.org/works/{main_work_id}"

# Step 1: Get all referenced work IDs
response = requests.get(main_url)
if response.status_code != 200:
    raise Exception(f"Error fetching main work: {response.status_code}")

data = response.json()
referenced_works = data.get("referenced_works", [])

print(f"Total referenced works found: {len(referenced_works)}")

# Step 2: Fetch titles for each referenced work
results = []

for ref_url in referenced_works:
    ref_id = ref_url.split("/")[-1]
    ref_api_url = f"https://api.openalex.org/works/{ref_id}"
    ref_response = requests.get(ref_api_url)

    if ref_response.status_code == 200:
        ref_data = ref_response.json()
        title = ref_data.get("display_name", "Title not found")
    else:
        title = f"Error {ref_response.status_code}"

    results.append((ref_id, title))
    time.sleep(0.2)  # optional: to respect rate limits

# Step 3: Print results
for rid, title in results:
    print(f"{rid}: {title}")


Total referenced works found: 89
W129102468: Mirtazapine vs. amitriptyline vs. placebo in the treatment of major depressive disorder.
W1904816587: Sertraline versus other antidepressive agents for depression
W1966406045: Benefits and harms in clinical trials of duloxetine for treatment of major depressive disorder: comparison of clinical study reports, trial registries, and publications
W1969298821: The cost of dichotomising continuous variables
W1971441597: Trial sequential analysis may establish when firm evidence is reached in cumulative meta-analysis
W1976275668: A meta-analysis of clinical trials comparing mirtazapine with selective serotonin reuptake inhibitors for the treatment of major depressive disorder
W1979281142: A double-blind, placebo-controlled study of antidepressant augementation with mirtazapine
W1979407421: Double-blind study of mirtazapine and placebo in hospitalized patients with major depression
W1988690727: Quality of life in major depressive disorder before/aft

In [14]:
import requests

# Constants
BASE_URL = "https://api.openalex.org/works"
CITED_WORK_ID = "W4406696697"

params = {
    "filter": f"cites:{CITED_WORK_ID}",
    "per-page": 200  # Max results per page
}

# Store (ID, title) pairs
citations = []

# Cursor-based pagination
next_page = BASE_URL
while next_page:
    response = requests.get(next_page, params=params if next_page == BASE_URL else None)
    if response.status_code != 200:
        print("Error fetching:", response.status_code)
        break

    data = response.json()
    for work in data.get("results", []):
        openalex_id = work["id"].split("/")[-1]
        title = work["display_name"]
        citations.append((openalex_id, title))

    # Pagination: update cursor
    cursor = data.get("meta", {}).get("next_cursor")
    next_page = f"{BASE_URL}?filter=cites:{CITED_WORK_ID}&cursor={cursor}" if cursor else None

# Print result: just OpenAlex ID and title
print(f"Total cited works found: {len(citations)}")

for wid, title in citations:
    print(f"{wid}: {title}")


Total cited works found: 1
W4407047355: Mirtazapine may not increase the risk of suicide in patients with MDD
